In [11]:
# ============================================================
# HMM POS TAGGER USING VITERBI ALGORITHM
# Dataset: Annotated Dataset for POS Tagging (Kaggle)
# ============================================================

import json
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)


# ============================================================
# 1. LOAD DATASET
# ============================================================

def load_dataset(file_path):

    sentences = []

    with open(file_path, "r", encoding="utf-8") as file:

        data = json.load(file)

        for item in data:

            words = item["sentence"]
            labels = item["labels"]

            sentences.append(list(zip(words, labels)))

    return sentences



# ============================================================
# DATASET PATH (YOUR PATH)
# ============================================================

train_path = r"C:\Users\acer\OneDrive\downloads\Sherin 5-sem NLP\NLP skill\CLASS TASK\POS\train.json"

dev_path = r"C:\Users\acer\OneDrive\downloads\Sherin 5-sem NLP\NLP skill\CLASS TASK\POS\dev.json"



train_sentences = load_dataset(train_path)

test_sentences = load_dataset(dev_path)



print("Training sentences:", len(train_sentences))

print("Testing sentences :", len(test_sentences))


print("\nSample:")
print(train_sentences[0])



# ============================================================
# 2. CREATE WORD AND TAG INDEX
# ============================================================

word_set = set()

tag_set = set()


for sentence in train_sentences:

    for word, tag in sentence:

        word_set.add(word)

        tag_set.add(tag)



words = list(word_set)

tags = list(tag_set)



word_to_index = {
    word:i for i,word in enumerate(words)
}


tag_to_index = {
    tag:i for i,tag in enumerate(tags)
}


index_to_tag = {
    i:tag for tag,i in tag_to_index.items()
}



num_words = len(words)

num_tags = len(tags)



print("\nVocabulary Size:", num_words)

print("Number of Tags:", num_tags)



# ============================================================
# 3. INITIAL PROBABILITY MATRIX
# ============================================================

initial_counts = np.ones(num_tags)


for sentence in train_sentences:

    first_tag = sentence[0][1]

    initial_counts[
        tag_to_index[first_tag]
    ] += 1



initial_probability = (
    initial_counts /
    initial_counts.sum()
)



# ============================================================
# 4. TRANSITION PROBABILITY MATRIX
# ============================================================

transition_counts = np.ones(
    (num_tags,num_tags)
)


for sentence in train_sentences:

    tag_sequence = [
        tag for word,tag in sentence
    ]


    for i in range(len(tag_sequence)-1):

        current_tag = tag_to_index[
            tag_sequence[i]
        ]

        next_tag = tag_to_index[
            tag_sequence[i+1]
        ]


        transition_counts[
            current_tag,
            next_tag
        ] += 1



transition_probability = (
    transition_counts /
    transition_counts.sum(axis=1, keepdims=True)
)



# ============================================================
# 5. EMISSION PROBABILITY MATRIX
# ============================================================

emission_counts = np.ones(
    (num_tags,num_words)
)



for sentence in train_sentences:

    for word,tag in sentence:

        emission_counts[
            tag_to_index[tag],
            word_to_index[word]
        ] += 1




emission_probability = (
    emission_counts /
    emission_counts.sum(axis=1, keepdims=True)
)



# ============================================================
# 6. CONVERT TO LOG SPACE
# ============================================================

log_initial = np.log(initial_probability)

log_transition = np.log(transition_probability)

log_emission = np.log(emission_probability)



# ============================================================
# 7. VECTORISED VITERBI ALGORITHM
# ============================================================

def viterbi(sentence):

    length = len(sentence)


    dp = np.full(
        (num_tags,length),
        -np.inf
    )


    backpointer = np.zeros(
        (num_tags,length),
        dtype=int
    )



    # First word

    word = sentence[0]


    if word in word_to_index:

        emission = log_emission[
            :,
            word_to_index[word]
        ]

    else:

        emission = np.log(
            np.ones(num_tags)/num_tags
        )



    dp[:,0] = (
        log_initial +
        emission
    )



    # Remaining words

    for t in range(1,length):

        word = sentence[t]


        if word in word_to_index:

            emission = log_emission[
                :,
                word_to_index[word]
            ]

        else:

            emission = np.log(
                np.ones(num_tags)/num_tags
            )



        scores = (
            dp[:,t-1][:,None]
            +
            log_transition
        )


        backpointer[:,t] = np.argmax(
            scores,
            axis=0
        )


        dp[:,t] = (
            np.max(scores,axis=0)
            +
            emission
        )



    # Backtracking

    best_tag = np.argmax(dp[:,-1])


    result = [best_tag]


    for t in range(length-1,0,-1):

        best_tag = backpointer[
            best_tag,
            t
        ]

        result.append(best_tag)



    result.reverse()


    return [
        index_to_tag[i]
        for i in result
    ]



# ============================================================
# 8. EVALUATION
# ============================================================

actual = []

predicted = []


for sentence in test_sentences:


    words_only = [
        word for word,tag in sentence
    ]


    real_tags = [
        tag for word,tag in sentence
    ]


    pred_tags = viterbi(words_only)



    actual.extend(real_tags)

    predicted.extend(pred_tags)



print("\n========== MODEL RESULTS ==========")


print(
    "Accuracy:",
    accuracy_score(actual,predicted)
)


print(
    "Precision:",
    precision_score(
    actual,
    predicted,
    average="weighted",
    zero_division=0
)
)


print(
    "Recall:",
    recall_score(
    actual,
    predicted,
    average="weighted",
    zero_division=0
)
)


print(
    "F1 Score:",
    f1_score(
    actual,
    predicted,
    average="weighted",
    zero_division=0
)
)



print("\nClassification Report:")

print(
    classification_report(
    actual,
    predicted,
    zero_division=0
)
)



# ============================================================
# 9. TEST FIVE UNSEEN SENTENCES
# ============================================================


new_sentences = [

    ["The","cat","is","sleeping"],

    ["She","writes","beautiful","poems"],

    ["Artificial","Intelligence","is","powerful"],

    ["Students","learn","Python"],

    ["Birds","fly","high"]

]



print("\n========== UNSEEN SENTENCE PREDICTION ==========")



for sentence in new_sentences:

    print("\nSentence:", sentence)

    print(
        "Predicted Tags:",
        viterbi(sentence)
    )

Training sentences: 38218
Testing sentences : 5527

Sample:
[('Pierre', 'NNP'), ('Vinken', 'NNP'), (',', ','), ('61', 'CD'), ('years', 'NNS'), ('old', 'JJ'), (',', ','), ('will', 'MD'), ('join', 'VB'), ('the', 'DT'), ('board', 'NN'), ('as', 'IN'), ('a', 'DT'), ('nonexecutive', 'JJ'), ('director', 'NN'), ('Nov.', 'NNP'), ('29', 'CD'), ('.', '.')]

Vocabulary Size: 43193
Number of Tags: 45

========== MODEL RESULTS ==========
Accuracy: 0.9241697528990347
Precision: 0.924066595308738
Recall: 0.9241697528990347
F1 Score: 0.921650454764807

Classification Report:
              precision    recall  f1-score   support

           #       1.00      0.94      0.97        31
           $       0.92      1.00      0.96      1144
          ''       0.86      1.00      0.92       991
           ,       0.98      1.00      0.99      7087
       -LRB-       1.00      0.93      0.96       181
       -RRB-       0.98      0.96      0.97       181
           .       0.96      1.00      0.98      5468
  